# Type 2 Diabetes Cohort Definition

This notebook demonstrates how to:
1. Manually define Type 2 Diabetes concepts
2. Build concept sets using CIRCE objects
3. Create a cohort definition using CIRCE Python
4. Generate SQL for OMOP CDM execution
5. Validate the cohort definition

## Requirements

```bash
pip install ohdsi-circe pandas
```


## Step 1: Import Required Libraries


In [2]:
# Core libraries
import pandas as pd
from IPython.display import display, Markdown


# CIRCE Python for cohort definitions
from circe.cohortdefinition import (
    CohortExpression, PrimaryCriteria, ConditionOccurrence,
    ObservationFilter, ResultLimit
)
from circe.vocabulary import ConceptSet, ConceptSetExpression, ConceptSetItem, Concept
from circe.api import build_cohort_query
from circe.check import Checker

print("✓ All libraries imported successfully")


✓ All libraries imported successfully


## Step 2: Helper Functions for ATHENA → CIRCE Conversion

Create efficient conversion functions to transform athena-client objects into CIRCE Concept and ConceptSet objects.


## Step 3: Define Type 2 Diabetes Concepts

Define the Concept Set for Type 2 Diabetes manually.


In [3]:
print("Defining Type 2 Diabetes concepts...")

# Manually create concept set
t2dm_concept_set = ConceptSet(
    id=1,
    name="Type 2 Diabetes Mellitus",
    expression=ConceptSetExpression(
        items=[
            ConceptSetItem(
                concept=Concept(
                    concept_id=201826,
                    concept_name="Type 2 diabetes mellitus",
                    vocabulary_id="SNOMED",
                    domain_id="Condition",
                    concept_class_id="Disorder",
                    standard_concept="S",
                    concept_code="44054006"
                ),
                include_descendants=True
            )
        ]
    )
)

print(f"\u2713 Concept set created:")
print(f"  ID: {t2dm_concept_set.id}")
print(f"  Name: {t2dm_concept_set.name}")
print(f"  Items: {len(t2dm_concept_set.expression.items)}")


Searching for Type 2 Diabetes concepts...
Searching ATHENA for 'Type 2 diabetes mellitus'...
✓ Found concepts
✓ Concept set created:
  ID: 1
  Name: Type 2 Diabetes Mellitus
  Items: 1
  Primary concept: 201826 - Type 2 diabetes mellitus
  Vocabulary: SNOMED


## Step 4: Verified Concept Set

The concept set is now ready for use in the cohort definition.


In [4]:
# Concept set created in previous step


## Step 5: Define Primary Criteria

Define the cohort entry criteria: first occurrence of Type 2 Diabetes condition.


In [5]:
# Create primary criteria: First Type 2 Diabetes diagnosis
primary_criteria = PrimaryCriteria(
    criteria_list=[
        ConditionOccurrence(
            codeset_id=1,                    # References concept set ID 1
            first=True,                      # Only the first occurrence
            condition_type_exclude=False     # Include all condition types
        )
    ],
    observation_window=ObservationFilter(
        prior_days=365,   # Patient must have 1 year observation starting on or before diagnosis
        post_days=1     # Patient must have observation on or after diagnosis
    ),
    primary_limit=ResultLimit(
        type="All"      # Include all qualifying events
    )
)

print("✓ Primary criteria defined:")
print(f"  Criteria Type: Condition Occurrence")
print(f"  Codeset ID: {primary_criteria.criteria_list[0].codeset_id}")
print(f"  First Occurrence Only: {primary_criteria.criteria_list[0].first}")


✓ Primary criteria defined:
  Criteria Type: Condition Occurrence
  Codeset ID: 1
  First Occurrence Only: True


## Step 6: Create Complete Cohort Expression

Combine concept sets and primary criteria into a complete cohort definition.


In [6]:
# Create the cohort expression
cohort = CohortExpression(
    title="Type 2 Diabetes Mellitus Patients",
    concept_sets=[t2dm_concept_set],
    primary_criteria=primary_criteria
)

print("✓ Cohort expression created:")
print(f"  Title: {cohort.title}")
print(f"  Number of Concept Sets: {len(cohort.concept_sets)}")
print(f"  Primary Criteria Type: {type(primary_criteria.criteria_list[0]).__name__}")


✓ Cohort expression created:
  Title: Type 2 Diabetes Mellitus Patients
  Number of Concept Sets: 1
  Primary Criteria Type: ConditionOccurrence


## Step 7: Validate Cohort Definition

Check for potential issues before generating SQL.


In [7]:
# Validate the cohort
checker = Checker()
warnings = checker.check(cohort)

if not warnings:
    print("✓ Cohort definition is valid with no warnings!")
else:
    print(f"⚠️ Validation found {len(warnings)} issues:")
    for warning in warnings:
        print(f"  {warning.to_message()}")


⚠️ Validation found 1 issues:
  It's not specified what type of records to look for in condition occurrence at initial event


## Step 8: Generate SQL Query

Generate the SQL query that can be executed against an OMOP CDM database.


In [8]:
# Generate SQL with your database schema names
from circe.cohortdefinition import BuildExpressionQueryOptions

options = BuildExpressionQueryOptions()
options.cdm_schema = "my_cdm_schema"        # Replace with your CDM schema name
options.vocabulary_schema = "my_vocab_schema"    # Replace with your vocabulary schema name
options.target_table = "cohort"
options.cohort_id = 1                        # Cohort ID for the results table

sql = build_cohort_query(cohort, options)

print(f"✓ SQL generated ({len(sql)} characters)")
print("\nFirst 1000 characters of SQL:")
print("=" * 80)
print(sql[:1000])
print("...")
print("=" * 80)


✓ SQL generated (5303 characters)

First 1000 characters of SQL:
CREATE TABLE #Codesets (
  codeset_id int NOT NULL,
  concept_id bigint NOT NULL
)
;

INSERT INTO #Codesets (codeset_id, concept_id)
SELECT 1 as codeset_id, c.concept_id FROM (select distinct I.concept_id FROM
( 
  select concept_id from my_vocab_schema.CONCEPT where (concept_id in (201826))
UNION  select c.concept_id
  from my_vocab_schema.CONCEPT c
  join my_vocab_schema.CONCEPT_ANCESTOR ca on c.concept_id = ca.descendant_concept_id
  WHERE c.invalid_reason is null
  and (ca.ancestor_concept_id in (201826))

) I
) C;

UPDATE STATISTICS #Codesets;


SELECT event_id, person_id, start_date, end_date, op_start_date, op_end_date, visit_occurrence_id
INTO #qualified_events
FROM 
(
  select pe.event_id, pe.person_id, pe.start_date, pe.end_date, pe.op_start_date, pe.op_end_date, row_number() over (partition by pe.person_id order by pe.start_date ASC) as ordinal, cast(pe.visit_occurrence_id as bigint) as visit_occurrence_id
  FR

## Step 9: Save Outputs

Save the cohort definition (JSON) and SQL query to files.


In [9]:
# Save cohort definition as JSON
# Note: For ATLAS compatibility, concepts should have complete metadata (see ATLAS_COMPATIBILITY.md)
cohort_json = cohort.model_dump_json(indent=2, by_alias=True, exclude_none=True)

with open('type2_diabetes_cohort.json', 'w') as f:
    f.write(cohort_json)
print("✓ Cohort definition saved to: type2_diabetes_cohort.json (ATLAS-compatible)")

# Save SQL query
with open('type2_diabetes_cohort.sql', 'w') as f:
    f.write(sql)
print("✓ SQL query saved to: type2_diabetes_cohort.sql")

# Display summary
print(f"\n{'='*80}")
print("SUMMARY")
print(f"{'='*80}")
print(f"Cohort Title: {cohort.title}")
print(f"Concept Sets: {len(cohort.concept_sets)}")
print(f"  - {t2dm_concept_set.name} (ID: {t2dm_concept_set.id})")
print(f"Primary Criteria: First Condition Occurrence")
print(f"SQL Length: {len(sql)} characters")
print(f"Validation: {'✓ PASSED' if not warnings else f'⚠️  {len(warnings)} warnings'}")
print(f"{'='*80}")


✓ Cohort definition saved to: type2_diabetes_cohort.json (ATLAS-compatible)
✓ SQL query saved to: type2_diabetes_cohort.sql

SUMMARY
Cohort Title: Type 2 Diabetes Mellitus Patients
Concept Sets: 1
  - Type 2 Diabetes Mellitus (ID: 1)
Primary Criteria: First Condition Occurrence
SQL Length: 5303 characters
Validation: ⚠️  1 warnings


In [10]:
cohort_json

'{\n  "ConceptSets": [\n    {\n      "id": 1,\n      "name": "Type 2 Diabetes Mellitus",\n      "expression": {\n        "isExcluded": false,\n        "includeMapped": false,\n        "includeDescendants": false,\n        "items": [\n          {\n            "concept": {\n              "CONCEPT_ID": 201826,\n              "CONCEPT_NAME": "Type 2 diabetes mellitus",\n              "CONCEPT_CODE": "44054006",\n              "CONCEPT_CLASS_ID": "Disorder",\n              "STANDARD_CONCEPT": "S",\n              "DOMAIN_ID": "Condition",\n              "VOCABULARY_ID": "SNOMED"\n            },\n            "isExcluded": false,\n            "includeMapped": false,\n            "includeDescendants": true\n          }\n        ]\n      }\n    }\n  ],\n  "PrimaryCriteria": {\n    "CriteriaList": [\n      {\n        "ConditionOccurrence": {\n          "CodesetId": 1,\n          "First": true,\n          "ConditionTypeExclude": false\n        }\n      }\n    ],\n    "ObservationWindow": {\n      

## Bonus: Manual Concept Set Creation

You can also manually define complex concept sets.


In [11]:
print("Defining Metformin concepts...")

metformin_concept_set = ConceptSet(
    id=2,
    name="Metformin",
    expression=ConceptSetExpression(
        items=[
            ConceptSetItem(
                concept=Concept(
                    concept_id=1125315,
                    concept_name="metformin",
                    vocabulary_id="RxNorm",
                    domain_id="Drug",
                    concept_class_id="Ingredient",
                    standard_concept="S",
                    concept_code="6809"
                ),
                include_descendants=True
            )
        ]
    )
)

print(f"\n\u2713 Metformin concept set created:")
print(f"  ID: {metformin_concept_set.id}")
print(f"  Name: {metformin_concept_set.name}")


Searching for Metformin concepts...
Searching ATHENA for 'Metformin'...
✓ Found concepts

✓ Metformin concept set created:
  ID: 2
  Name: Metformin
    - 1125315: metformin


## Optional: Specifying Condition Types (to suppress INFO warning)

If you want to be explicit about which condition record types to include (and suppress the validation INFO message), you can specify `condition_type`. By default, leaving it `None` means "accept all types".


In [12]:
# Example: Create a more specific cohort that only accepts EHR records
# (This will have zero validation warnings)

from circe.cohortdefinition import ConditionOccurrence

# Common condition type concepts (from OMOP CDM concept table)
# These specify where the condition diagnosis came from:
# - 32020: EHR problem list entry
# - 32817: EHR
# - 32809: Case Report Form
# - 32810: Patient self-report

ehr_condition_types = [
    Concept(
        concept_id=32817,
        concept_name="EHR",
        domain_id="Type Concept",
        vocabulary_id="Condition Type",
        concept_class_id="Condition Type",
        standard_concept="S",
        concept_code="OMOP4976890"
    )
]

# Create criteria with explicit condition type
specific_criteria = PrimaryCriteria(
    criteria_list=[
        ConditionOccurrence(
            codeset_id=1,
            first=True,
            condition_type=ehr_condition_types,  # Specify EHR records only
            condition_type_exclude=False
        )
    ],
    observation_window=ObservationFilter(prior_days=365, post_days=0),
    primary_limit=ResultLimit(type="All")
)

# Create cohort with specific criteria
specific_cohort = CohortExpression(
    title="Type 2 Diabetes (EHR Only)",
    concept_sets=[t2dm_concept_set],
    primary_criteria=specific_criteria
)

# Validate - should have zero warnings now
checker2 = Checker()
warnings2 = checker2.check(specific_cohort)

if not warnings2:
    print("✓ Specific cohort has NO warnings!")
else:
    print(f"⚠️ Found {len(warnings2)} warnings:")
    for w in warnings2:
        print(f"  [{w.severity.name}] {w.message}")

print(f"\nℹ️  Note: Using condition_type is optional. The original cohort")
print(f"   (without condition_type) will work perfectly fine - it just accepts")
print(f"   ALL condition types, which is usually what you want.")


✓ Specific cohort has NO warnings!

ℹ️  Note: Using condition_type is optional. The original cohort
   (without condition_type) will work perfectly fine - it just accepts
   ALL condition types, which is usually what you want.


## Summary

This notebook demonstrated:

1. ✓ Creating Concept Sets manually
2. ✓ Building concept sets using CIRCE objects
3. ✓ Creating a cohort definition with CIRCE Python
4. ✓ Validating the cohort definition
5. ✓ Generating SQL for OMOP CDM execution
6. ✓ Exporting cohort definition as JSON

### Files Created

- `type2_diabetes_cohort.json` - Cohort definition (Atlas-compatible)
- `type2_diabetes_cohort.sql` - SQL query for OMOP CDM

### Additional Resources

- **CIRCE Python docs**: https://ohdsi-circe.readthedocs.io/
- **OHDSI ATHENA**: https://athena.ohdsi.org/
- **OMOP CDM**: https://ohdsi.github.io/CommonDataModel/
